# Retrieval Quality Analyzer

**The question this tool answers:** *"When my AI searches my documents, does it find the RIGHT one?"*

Most AI apps that answer from your own documents use **RAG** (Retrieval-Augmented Generation): before the AI writes anything, the system **retrieves** the most relevant document and hands it over. If retrieval grabs the *wrong* document, the AI writes a confident answer from the wrong source — and this failure is invisible unless you measure the retrieval step directly.

This tool measures retrieval quality and then experiments to improve it.

### The one new idea: embeddings
A computer can't tell that *"money back"* and *"refund"* mean the same thing — they share no words. An **embedding** converts text into a list of numbers (a "vector") that captures its *meaning*, so similar meanings get similar numbers. Retrieval then = "find the documents whose numbers are closest to the question's numbers." Search by **meaning**, not by matching words.

> Note: retrieval needs **no LLM and no API key** — it's a separate step *before* the AI writes. We use a small, free embedding model that runs right here in Colab.

Run cells top to bottom with `Shift + Enter`.

## Step 1 — Install tools

- `sentence-transformers` — a free library that turns text into embeddings, running locally (no API key).
- `scikit-learn` — for measuring how close two vectors are.
- `duckdb`, `pandas` — storage and tables, as in the earlier projects.

The first install pulls a small model, so this cell takes a minute.

In [ ]:
!pip install sentence-transformers scikit-learn duckdb pandas -q
print("Installed.")

## Step 2 — Load the embedding model

`all-MiniLM-L6-v2` is a small, fast, well-known embedding model. It downloads once (~90 MB) and then runs on your Colab machine. This is the thing that converts text → meaning-numbers.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model ready.")

# Quick demo: two differently-worded but similar sentences get similar vectors.
import numpy as np
a = embedder.encode("money back for my purchase")
b = embedder.encode("refund policy for orders")
c = embedder.encode("how long does shipping take")
def cos(x, y): return float(np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y)))
print(f"'money back' vs 'refund'  similarity: {cos(a, b):.2f}  (high — similar meaning)")
print(f"'money back' vs 'shipping' similarity: {cos(a, c):.2f}  (low  — different meaning)")

## Step 3 — Load the knowledge base and the labeled questions

Two parts:
- **documents** — a small support knowledge base (10 short articles).
- **questions** — each with the `relevant_doc` that *should* be retrieved. This label is what lets us grade retrieval: we know the right answer, so we can check whether the system found it.

The questions deliberately use *different words* than the documents (e.g. "Does this work on my iPhone?" for the doc that says "iOS"). Word-matching would fail these; embeddings should handle them.

In [ ]:
dataset = {
  "documents": [
    {
      "doc_id": "d01",
      "title": "Password Reset",
      "text": "To reset your password, go to the login page and click 'Forgot Password'. Enter your registered email address and we will send a reset link valid for 30 minutes. If you do not receive the email, check your spam folder. For security, the link can only be used once. After resetting, you will be logged out of all devices and must sign in again with your new password."
    },
    {
      "doc_id": "d02",
      "title": "Refund Policy",
      "text": "We offer full refunds within 30 days of purchase for unused products in their original packaging. Refunds are processed to the original payment method within 5 to 7 business days. Digital products are non-refundable once downloaded. Sale items marked as final sale cannot be returned. To start a refund, contact support with your order number."
    },
    {
      "doc_id": "d03",
      "title": "Shipping Times",
      "text": "Standard shipping takes 3 to 5 business days within the continental United States. Express shipping is available for an additional fee and takes 1 to 2 business days. International shipping takes 7 to 14 business days depending on the destination country and local customs processing. Tracking numbers are emailed once your order ships."
    },
    {
      "doc_id": "d04",
      "title": "Subscription Plans",
      "text": "You can upgrade or downgrade your subscription plan at any time from your account settings. Upgrades take effect immediately and you are charged a prorated amount for the remainder of the billing cycle. Downgrades take effect at the start of your next billing cycle. Annual plans receive a 20 percent discount compared to monthly billing."
    },
    {
      "doc_id": "d05",
      "title": "Payment Security",
      "text": "All payment information is encrypted using industry-standard TLS encryption. We do not store your full card number on our servers. Payments are processed through PCI-DSS compliant third-party providers. We support credit cards, debit cards, and digital wallets. Fraudulent activity is monitored continuously and suspicious transactions may be held for review."
    },
    {
      "doc_id": "d06",
      "title": "Contact Support",
      "text": "Our customer support team is available Monday through Friday, 9 AM to 6 PM Eastern Time. You can reach us by email, by phone, or through live chat on our website during business hours. Enterprise customers have access to a dedicated priority support line and a named account manager for faster resolution."
    },
    {
      "doc_id": "d07",
      "title": "Free Trial",
      "text": "New users can start a 14-day free trial with full access to all premium features. No credit card is required to begin the trial. At the end of the trial, you can choose a paid plan or your account will automatically switch to the limited free tier. Only one free trial is allowed per customer."
    },
    {
      "doc_id": "d08",
      "title": "Account Cancellation",
      "text": "To cancel your account, go to account settings and select 'Cancel Account'. Your account remains active until the end of your current billing period, so you keep access to paid features until then. After cancellation, your data is retained for 90 days in case you wish to reactivate, after which it is permanently deleted."
    },
    {
      "doc_id": "d09",
      "title": "Mobile App",
      "text": "Our mobile app is available for both iOS and Android devices. It offers the same core features as the web version, plus push notifications and offline access to saved items. The app requires iOS 15 or later, or Android 10 or later. Updates are released monthly with new features and bug fixes."
    },
    {
      "doc_id": "d10",
      "title": "Data Export",
      "text": "You can export your data at any time from the privacy section of your account settings. Exports are provided as a downloadable file in CSV or JSON format. Large exports may take up to 24 hours to prepare, and you will receive an email with a secure download link when your export is ready."
    }
  ],
  "questions": [
    {
      "q_id": "q01",
      "question": "How do I change my password if I forgot it?",
      "relevant_doc": "d01"
    },
    {
      "q_id": "q02",
      "question": "Can I get my money back for something I bought?",
      "relevant_doc": "d02"
    },
    {
      "q_id": "q03",
      "question": "How long until my order arrives?",
      "relevant_doc": "d03"
    },
    {
      "q_id": "q04",
      "question": "What happens if I switch to a cheaper plan?",
      "relevant_doc": "d04"
    },
    {
      "q_id": "q05",
      "question": "Is it safe to enter my credit card details?",
      "relevant_doc": "d05"
    },
    {
      "q_id": "q06",
      "question": "When can I talk to a human for help?",
      "relevant_doc": "d06"
    },
    {
      "q_id": "q07",
      "question": "Do I need to pay to try the product first?",
      "relevant_doc": "d07"
    },
    {
      "q_id": "q08",
      "question": "How do I close my account for good?",
      "relevant_doc": "d08"
    },
    {
      "q_id": "q09",
      "question": "Does this work on my iPhone?",
      "relevant_doc": "d09"
    },
    {
      "q_id": "q10",
      "question": "How can I download a copy of all my information?",
      "relevant_doc": "d10"
    }
  ]
}

documents = dataset["documents"]
questions = dataset["questions"]
print(f"Loaded {len(documents)} documents and {len(questions)} labeled questions.")

## Step 4 — Chunking: the thing we'll experiment with

Real documents are long, so RAG systems split them into **chunks** and embed each chunk separately. **Chunk size matters enormously** and it's the #1 mistake people make.

- **Big chunks** (whole document): each embedding has to represent many topics at once, so its meaning gets "blurry" and retrieval is less precise.
- **Small chunks** (a sentence or two): each embedding captures one focused idea, so retrieval is sharper.

Below, `chunk_text` splits a document into pieces of roughly `chunk_size` words. We'll build the search index twice — once with big chunks, once with small — and compare. Each chunk remembers which document it came from (`doc_id`), so retrieving a chunk tells us which document was found.

In [ ]:
def chunk_text(text, chunk_size):
    """Split text into chunks of about chunk_size words."""
    words = text.split()
    chunks = []
    for start in range(0, len(words), chunk_size):
        piece = " ".join(words[start:start + chunk_size])
        chunks.append(piece)
    return chunks

def build_chunks(documents, chunk_size):
    """Turn all documents into a flat list of chunks, each tagged with its doc_id."""
    all_chunks = []
    for doc in documents:
        for piece in chunk_text(doc["text"], chunk_size):
            all_chunks.append({"doc_id": doc["doc_id"], "text": piece})
    return all_chunks

# Peek: how many chunks does each strategy produce?
big = build_chunks(documents, 1000)   # 1000 words = effectively "whole document"
small = build_chunks(documents, 25)   # ~25 words = a sentence or two
print(f"Big-chunk strategy:   {len(big)} chunks (about one per document)")
print(f"Small-chunk strategy: {len(small)} chunks (many focused pieces)")

## Step 5 — The retrieval function

This is the search itself. Steps:
1. Embed every chunk (convert each to its meaning-numbers) — this is the "index".
2. Embed the question.
3. Compute similarity between the question and every chunk.
4. Return the `top_k` closest chunks — and therefore which documents they came from.

`cosine_similarity` is just the standard way to measure how close two vectors are (1.0 = identical meaning, 0 = unrelated).

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def build_index(chunks):
    """Embed all chunks once. Returns the matrix of vectors."""
    texts = [c["text"] for c in chunks]
    vectors = embedder.encode(texts)
    return vectors

def retrieve(question, chunks, index, top_k):
    """Return the doc_ids of the top_k most similar chunks to the question."""
    q_vec = embedder.encode([question])
    sims = cosine_similarity(q_vec, index)[0]        # similarity to every chunk
    top_idx = np.argsort(sims)[::-1][:top_k]         # positions of the highest scores
    return [chunks[i]["doc_id"] for i in top_idx]

## Step 6 — The retrieval metrics

We grade retrieval with two standard measures. For each question we know the one correct document (`relevant_doc`):

- **Hit@k (a form of recall):** did the correct document appear anywhere in the top-k results? 1 if yes, 0 if no. Averaged over all questions, it's "how often did we find the right doc at all."
- **MRR (Mean Reciprocal Rank):** *where* did the correct document rank? If it was 1st → score 1.0; 2nd → 0.5; 3rd → 0.33; not found → 0. This rewards putting the right doc at the *top*, not just somewhere in the list.

Both range 0 to 1, higher is better.

In [ ]:
def evaluate(chunks, index, top_k):
    """Run every question through retrieval and compute Hit@k and MRR."""
    hits = 0
    reciprocal_ranks = 0.0
    per_question = []

    for item in questions:
        retrieved = retrieve(item["question"], chunks, index, top_k)
        correct = item["relevant_doc"]

        # de-duplicate doc_ids while keeping order (several chunks may share a doc)
        seen = []
        for d in retrieved:
            if d not in seen:
                seen.append(d)

        if correct in seen:
            hits += 1
            rank = seen.index(correct) + 1        # 1-based position
            reciprocal_ranks += 1.0 / rank
            rr = 1.0 / rank
        else:
            rr = 0.0

        per_question.append({
            "q_id": item["q_id"],
            "found": correct in seen,
            "reciprocal_rank": round(rr, 3),
            "top_docs": seen[:top_k],
        })

    n = len(questions)
    return {
        "hit_rate": hits / n,
        "mrr": reciprocal_ranks / n,
        "per_question": per_question,
    }

## Step 7 — The experiment: big chunks vs. small chunks 🔬

Now the payoff. We build a search index for each strategy and evaluate both on the same questions with the same `top_k`. Watch the difference.

In [ ]:
TOP_K = 3   # retrieve the 3 closest chunks for each question

# Strategy A: big chunks (whole documents)
big_chunks = build_chunks(documents, 1000)
big_index = build_index(big_chunks)
big_result = evaluate(big_chunks, big_index, TOP_K)

# Strategy B: small chunks (~25 words)
small_chunks = build_chunks(documents, 25)
small_index = build_index(small_chunks)
small_result = evaluate(small_chunks, small_index, TOP_K)

print("BIG chunks   -> Hit@3: {:.2f}   MRR: {:.2f}".format(big_result["hit_rate"], big_result["mrr"]))
print("SMALL chunks -> Hit@3: {:.2f}   MRR: {:.2f}".format(small_result["hit_rate"], small_result["mrr"]))
print()
winner = "SMALL" if small_result["mrr"] > big_result["mrr"] else "BIG"
print(f"Better strategy: {winner} chunks")

## Step 8 — Inspect what each strategy got wrong

Averages hide detail. This shows, per question, whether each strategy found the right document and at what rank. Look for questions the big-chunk strategy missed (found = False) that the small-chunk strategy caught — those are the wins from better chunking.

In [ ]:
import pandas as pd

rows = []
for b, s in zip(big_result["per_question"], small_result["per_question"]):
    rows.append({
        "question": b["q_id"],
        "big_found": b["found"],
        "big_rank_score": b["reciprocal_rank"],
        "small_found": s["found"],
        "small_rank_score": s["reciprocal_rank"],
    })
pd.DataFrame(rows)

## Step 9 — Save results for the dashboard

We store both the summary (the head-to-head numbers) and the per-question detail in a DuckDB file for the Streamlit dashboard.

In [ ]:
import duckdb

con = duckdb.connect("retrieval_results.db")
con.execute("CREATE TABLE IF NOT EXISTS summary (strategy VARCHAR, hit_rate DOUBLE, mrr DOUBLE, top_k INTEGER)")
con.execute("CREATE TABLE IF NOT EXISTS per_question (strategy VARCHAR, q_id VARCHAR, found BOOLEAN, reciprocal_rank DOUBLE)")
con.execute("DELETE FROM summary")
con.execute("DELETE FROM per_question")

for name, res in [("Big chunks", big_result), ("Small chunks", small_result)]:
    con.execute("INSERT INTO summary VALUES (?, ?, ?, ?)", [name, res["hit_rate"], res["mrr"], TOP_K])
    for pq in res["per_question"]:
        con.execute("INSERT INTO per_question VALUES (?, ?, ?, ?)",
                    [name, pq["q_id"], pq["found"], pq["reciprocal_rank"]])

con.close()
print("Saved to retrieval_results.db")
print("Download it (Files panel) and drop it into the dashboard's data/ folder.")

## ✅ Done — what you built

A retrieval quality analyzer that:
- turns documents and questions into **embeddings** (meaning-vectors),
- **retrieves** the closest documents for each question,
- scores retrieval with **Hit@k** and **MRR**,
- and runs an **A/B experiment** (big vs. small chunks) showing which retrieves better.

**The takeaway to explain in interviews:** RAG systems usually fail at *retrieval*, not generation, and chunk size is a huge, often-overlooked lever. You didn't just build a RAG app — you built the tool that *diagnoses* why one retrieves badly. That's the more senior skill.

**To make it yours:** swap in your own documents and questions, or add a third strategy (e.g. a different `TOP_K`, or a different embedding model) to the experiment.

**Next:** download `retrieval_results.db` and open the Streamlit dashboard to see the comparison visually.